# E008: cheap validation -> one CV -> final fit -> test prediction

E004-E007 (blocking only) picked `base,conj:20,bm25:5,bge_native:5,name_noaddr:5`:
all-S1 recall **0.9818** (E003: 0.8610), oracle macro F0.5 **0.9942** (E003: 0.9338), 145M pairs.

| step | what | cost |
|---|---|---|
| 1 | **Holdout**: one split, E003 blocker vs E007 blocker, same sampled S1s. Train 70% / val-A 15% (early stop + threshold) / val-B 15% (reported) | 2 model runs |
| 2 | **CV once** (3 folds + country stress) on E007, only if step 1 shows a gain | optional |
| 3 | **Fit** final LightGBM (rounds and threshold from CV, else from the holdout) | 1 model |
| 4 | **Block test** with the E007 union (BGE-M3 needs the GPU), then **predict** + official validator | |

**Run this in the SAME Kaggle notebook as E004** (paste or import these cells there) so the cached
`work/prepared/train` and both train candidate sets are reused from `/kaggle/working`. Otherwise attach a
saved version of that notebook's output as input: the caches are picked up from `/kaggle/input` as well.

Settings: Accelerator **GPU T4 x2** (needed in step 4 for BGE-M3 on test). Internet **On**. Persistence **Files**.
Every step caches and is skipped when its output exists, so *Run all* resumes after a crash.

In [ ]:
# 1. Config
E003_EXP, E007_EXP = "20260925-E008-holdout-E003", "20260925-E008-E007"
E007_SPEC = "base,conj:20,bm25:5,bge_native:5,name_noaddr:5"
CV_FRAC   = 0.1      # sampled train S1 for features (~220k S1; E007 ~14.5M pairs)
FIT_FRAC  = 0.1      # raise to 0.2 for the final fit if the step-1 timings leave room
RUN_CV    = True     # one 3-fold CV + country stress on E007 (spec: only after the cheap validation)
LR        = 0.1
REPO      = "https://github.com/Bexwane/AmazonMLchallenge.git"
CODE_DIR  = "/kaggle/working/ber"
WORK      = "/kaggle/working/work"
OUT       = "/kaggle/working/output"

In [ ]:
# 2. Dataset, validator, and caches (from this notebook's /kaggle/working or from an attached saved output)
import glob, os
hits = glob.glob("/kaggle/input/**/train/train_source1.tsv", recursive=True)
assert hits, "Dataset not found: add the dataset with train/ and test/ folders as notebook input"
DATA = os.path.dirname(os.path.dirname(hits[0]))
val = glob.glob("/kaggle/input/**/validate_submission.py", recursive=True)
VALIDATOR = val[0] if val else None
def reuse(pattern, dest_dir):
    for p in glob.glob(f"/kaggle/input/**/{pattern}", recursive=True):
        d = os.path.join(dest_dir, os.path.basename(p))
        if not os.path.exists(d):
            os.makedirs(dest_dir, exist_ok=True); os.symlink(p, d); print("reusing", p)
for split in ("train", "test"):
    reuse(f"work/prepared/{split}/*.parquet", f"{WORK}/prepared/{split}")
    reuse(f"work/{split}/cand_*.parquet", f"{WORK}/{split}")
print("DATA =", DATA, "| VALIDATOR =", VALIDATOR)
!ls -la {WORK}/train {WORK}/prepared/train 2>/dev/null; free -g; nproc; nvidia-smi -L 2>/dev/null || echo "no GPU"

In [ ]:
# 3. Code, dependencies, tests
!rm -rf {CODE_DIR} && git clone -q {REPO} {CODE_DIR} && cd {CODE_DIR} && git log --oneline -1
!pip install -q rapidfuzz==3.14.6 && pip install -q -U sentence-transformers
import sys; sys.path.insert(0, f"{CODE_DIR}/src")
!cd {CODE_DIR} && python -m pytest -q tests

In [ ]:
# helper: run a pipeline command with live, timestamped output (exit -9 = out of memory)
import subprocess, time, json
import pandas as pd
def ber(cmd, exp, *extra, frac=CV_FRAC, spec=E007_SPEC):
    args = ["python", "-m", "ber.run", cmd, "--data", DATA, "--work", WORK, "--exp", exp, "--cv-frac", str(frac),
            "--lr", str(LR), "--df-cap", "2500", *(["--channels", spec] if spec else ["--k-comb", "45", "--k-name", "10"]),
            *map(str, extra)]
    t = time.time()
    p = subprocess.Popen(args, cwd=CODE_DIR, env={**os.environ, "PYTHONPATH": "src"},
                         stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)
    for line in p.stdout:
        print(line, end="")
    assert p.wait() == 0, f"{cmd} failed (exit {p.returncode}); -9 means out of memory"
    print(f"--- {cmd} done in {(time.time() - t) / 60:.1f} min")
def exp_file(exp, name):
    return f"{WORK}/experiments/{exp}/{name}"

In [ ]:
# 4. STEP 1a: blocking caches (skipped when cached; E007 train blocking takes ~1.6 h and needs the GPU)
ber("block", E003_EXP, "--split", "train", spec="")
ber("block", E007_EXP, "--split", "train")
!ls -la {WORK}/train

In [ ]:
# 5. STEP 1b: cheap validation, E003 blocker vs E007 blocker on the same sampled S1s
for exp, spec in ((E003_EXP, ""), (E007_EXP, E007_SPEC)):
    if not os.path.exists(exp_file(exp, "holdout_metrics.json")):
        ber("holdout", exp, spec=spec)
rows = []
for name, exp in (("E003 blocker", E003_EXP), ("E007 blocker", E007_EXP)):
    h = json.load(open(exp_file(exp, "holdout_metrics.json")))
    b = h["val_B"]
    rows.append({"blocker": name, "val_B macro F0.5": b["macro_f05"], "precision": b["micro_precision"],
                 "recall": b["micro_recall"], "singleton F0.5": b["f05_singletons"], "non-singleton F0.5": b["f05_nonsingletons"],
                 **{f"{c} F0.5": h[f"val_B_{c}"]["macro_f05"] for c in ("India", "US") if f"val_B_{c}" in h},
                 "threshold": h["threshold"], "best_iter": h["best_iter"],
                 "pair_recall (sampled)": h["blocking_sampled"]["pair_recall"]})
cmp = pd.DataFrame(rows).set_index("blocker").T
display(cmp.round(4))
GAIN = cmp.loc["val_B macro F0.5", "E007 blocker"] - cmp.loc["val_B macro F0.5", "E003 blocker"]
print(f"E007 - E003 on val-B: {GAIN:+.4f}")

In [ ]:
# 6. STEP 2: CV once on E007 (3 folds + country-holdout stress), only if the cheap validation shows a gain
if RUN_CV and GAIN > 0.002 and not os.path.exists(exp_file(E007_EXP, "cv_metrics.json")):
    ber("cv", E007_EXP, "--folds", 3)
if os.path.exists(exp_file(E007_EXP, "cv_metrics.json")):
    cv = json.load(open(exp_file(E007_EXP, "cv_metrics.json")))
    display(pd.DataFrame({k: v for k, v in cv.items() if k.startswith(("cv", "stress", "no_excl"))}).T.round(4))
    print("threshold", cv["threshold"], "best_iters", cv["best_iters"])

In [ ]:
# 7. STEP 3: final fit (rounds + threshold from CV if it ran, else from the holdout)
if not os.path.exists(exp_file(E007_EXP, "model.txt")) or FIT_FRAC != CV_FRAC:
    ber("fit", E007_EXP, frac=FIT_FRAC)
print(open(exp_file(E007_EXP, "fit.json")).read())

In [ ]:
# 8. STEP 4: test blocking (GPU for BGE-M3), prediction, and the official validator
ber("block", E007_EXP, "--split", "test")
ber("predict", E007_EXP, "--out", OUT, frac=FIT_FRAC)
MATCH, CAND = f"{OUT}/matching_results.tsv", f"{OUT}/candidate_pairs.tsv"
if VALIDATOR:
    !python {VALIDATOR} --matching {MATCH} --candidate {CAND} --test-dir {DATA}/test --check-ids
from ber.io import read_ground_truth, read_source
pred = read_ground_truth(MATCH)
s1 = read_source(f"{DATA}/test/test_source1.tsv")
s1["n"] = s1["entity_id"].map(lambda e: len(pred.get(e, ())))
display(s1.groupby("country")["n"].agg(mean_matches="mean", empty_rate=lambda x: (x == 0).mean(), n="size").round(3))
!ls -la {OUT}; du -sh {OUT}

In [ ]:
# 9. Bundle for review: send /kaggle/working/E008_results.tgz back (metrics only; the TSVs stay in output/)
import shutil, tarfile
B = "/kaggle/working/E008_results"
os.makedirs(B, exist_ok=True)
for exp in (E003_EXP, E007_EXP):
    for f in ("holdout_metrics.json", "cv_metrics.json", "fit.json"):
        if os.path.exists(exp_file(exp, f)):
            shutil.copy(exp_file(exp, f), f"{B}/{exp}_{f}")
with tarfile.open("/kaggle/working/E008_results.tgz", "w:gz") as t:
    t.add(B, arcname="E008_results")
print(sorted(os.listdir(B)))